# Stage 1 — Multilingual Baseline (O1-lang vs O1-bos)

Fine-tunes the English checkpoint on 7 languages with a **learned language embedding** prepended to the encoder input. Compares encoder-level vs decoder-level language steering.

## Setup

In [ ]:
import sys, os
sys.path.append("../src")

import torch
import torch.nn as nn
from transformers import AutoTokenizer
from models import ProjectionMLP, load_mt5_with_lora
from dataset import TRAINING_LANGUAGES, LANG_TO_ID, LANG_PREFIXES, build_multilingual_loaders

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = AutoTokenizer.from_pretrained("google/mt5-base")
print(f"Device: {device} | Languages: {TRAINING_LANGUAGES}")

## Load Models from English Checkpoint

In [ ]:
projection = ProjectionMLP().to(device)
mt5        = load_mt5_with_lora().to(device)
lang_emb   = nn.Embedding(len(TRAINING_LANGUAGES), 768).to(device)

ckpt = torch.load("../outputs/english_best.pt", weights_only=True)
projection.load_state_dict(ckpt["projection"])
mt5.load_state_dict(ckpt["lora"])
print("English checkpoint loaded ✓")

## Build Multilingual DataLoaders

Uses `WeightedRandomSampler` to balance across languages (prevents model collapsing to Vietnamese, the largest dataset).

In [ ]:
# Map each language to its precomputed feature file
feature_files = {lang: f"../{lang}_features.pt" for lang in TRAINING_LANGUAGES}

train_loader, val_loader = build_multilingual_loaders(
    feature_files = feature_files,
    tokenizer     = tokenizer,
    batch_size    = 16,
)
print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)}")

## Train O1-lang (encoder-level language embedding)

In [ ]:
EPOCHS    = 10
optimizer = torch.optim.AdamW(
    list(projection.parameters()) + list(mt5.parameters()) + list(lang_emb.parameters()),
    lr=1e-4, weight_decay=1e-2,
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
best_val  = float("inf")

for epoch in range(EPOCHS):
    projection.train(); mt5.train(); lang_emb.train()
    train_loss = 0

    from tqdm import tqdm
    for features, labels, lang_ids in tqdm(train_loader, desc=f"Epoch {epoch+1}", leave=False):
        features, labels, lang_ids = features.to(device), labels.to(device), lang_ids.to(device)

        # Prepend language token to visual features: (B, 33, 768)
        enc_input = torch.cat([lang_emb(lang_ids).unsqueeze(1), projection(features)], dim=1)
        masked_labels = labels.clone()
        masked_labels[masked_labels == tokenizer.pad_token_id] = -100

        loss = mt5(inputs_embeds=enc_input, labels=masked_labels).loss
        optimizer.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(
            list(projection.parameters()) + list(mt5.parameters()) + list(lang_emb.parameters()), 1.0
        )
        optimizer.step()
        train_loss += loss.item()

    # Validation
    projection.eval(); mt5.eval(); lang_emb.eval()
    val_loss = 0
    with torch.no_grad():
        for features, labels, lang_ids in val_loader:
            features, labels, lang_ids = features.to(device), labels.to(device), lang_ids.to(device)
            enc_input = torch.cat([lang_emb(lang_ids).unsqueeze(1), projection(features)], dim=1)
            masked_labels = labels.clone()
            masked_labels[masked_labels == tokenizer.pad_token_id] = -100
            val_loss += mt5(inputs_embeds=enc_input, labels=masked_labels).loss.item()

    train_loss /= len(train_loader)
    val_loss   /= len(val_loader)
    scheduler.step()
    print(f"Epoch {epoch+1:2d} | Train: {train_loss:.4f} | Val: {val_loss:.4f}")

    if val_loss < best_val:
        best_val = val_loss
        torch.save({"projection": projection.state_dict(), "lora": mt5.state_dict(),
                    "lang_emb": lang_emb.state_dict()},
                   "../outputs/multilingual_best.pt")
        print(f"  ✓ Saved (val={val_loss:.4f})")

## Evaluate on XM3600

Run the evaluation script from the terminal for full results:
```bash
python src/evaluate.py --stage multilingual --checkpoint outputs/multilingual_best.pt
```